In [3]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("API_KEY")

In [4]:
url = "https://api.semanticscholar.org/graph/v1/paper/search"

params = {
    "query": "machine learning",
    "limit": 5,
    "fields": "title,authors,year,citationCount,abstract,paperId"
}

headers = {
    "User-Agent": "Alyssa Wilson (flutelys@byu.edu)",
    "x-api-key": api_key
}

response = requests.get(url, params=params, headers=headers)
data = response.json()

print(data.keys())
print(data)

dict_keys(['total', 'offset', 'next', 'data'])
{'total': 7364557, 'offset': 0, 'next': 5, 'data': [{'paperId': '53c9f3c34d8481adaf24df3b25581ccf1bc53f5c', 'title': 'Physics-informed machine learning', 'year': 2021, 'citationCount': 6076, 'openAccessPdf': {'url': 'https://www.osti.gov/biblio/2282016', 'status': 'GREEN', 'license': 'other-oa', 'disclaimer': "Notice: The following paper fields have been elided by the publisher: {'abstract'}. Paper or abstract available at https://api.unpaywall.org/v2/10.1038/s42254-021-00314-5?email=<INSERT_YOUR_EMAIL> or https://doi.org/10.1038/s42254-021-00314-5, which is subject to the license by the author or copyright owner provided with this content. Please go to the source to verify the license and copyright information for your use."}, 'authors': [{'authorId': '1720124', 'name': 'G. Karniadakis'}, {'authorId': '3439407', 'name': 'I. Kevrekidis'}, {'authorId': '2149373363', 'name': 'Lu Lu'}, {'authorId': '3410970', 'name': 'P. Perdikaris'}, {'autho

In [13]:
import time
import requests

all_rows = []
search_terms = [
    'llm', 'CoT', 'reasoning', 'chain of thought', 'rag', 'grounding',
    'external memory', 'multimodal', 'image generation', 'rlhf',
    'lora', 'qlora', 'quantization', 'efficient fine-tuning',
    'long context and efficient', 'sparse attention', 'flash attention',
    'hallucination', 'efficient inference'
]

seen_ids = set()

# --- Global rate limiter ---
last_request_time = 0

def rate_limited_get(url, headers=None, params=None):
    global last_request_time
    
    elapsed = time.time() - last_request_time
    if elapsed < 1:
        time.sleep(1 - elapsed)
    
    response = requests.get(url, headers=headers, params=params)
    last_request_time = time.time()
    
    return response


for search_term in search_terms:
    for offset in range(0, 100, 10):  # adjust later if needed

        params = {
            "query": search_term,
            "limit": 10,
            "offset": offset,
            "fields": "title,authors,year,citationCount,abstract,influentialCitationCount,paperId"
        }

        # --- Search request ---
        response = rate_limited_get(url, headers=headers, params=params)

        if response.status_code != 200:
            print("Search error:", response.text)
            continue

        data = response.json()
        papers = data.get("data", [])

        if not papers:
            break

        for paper in papers:
            paper_id = paper.get("paperId")

            if paper_id in seen_ids:
                continue
            seen_ids.add(paper_id)

            # --- Get fields from search ---
            abstract = paper.get("abstract")
            influential = paper.get("influentialCitationCount")

            # --- Only fetch if abstract missing ---
            if abstract is None:
                paper_url = f"https://api.semanticscholar.org/graph/v1/paper/{paper_id}"
                paper_params = {"fields": "abstract"}

                paper_resp = rate_limited_get(
                    paper_url,
                    headers=headers,
                    params=paper_params
                )

                if paper_resp.status_code != 200:
                    print("Paper error:", paper_resp.text)
                    continue

                paper_data = paper_resp.json()
                abstract = paper_data.get("abstract")

            # --- Store result ---
            all_rows.append({
                "title": paper.get("title"),
                "authors": ", ".join(a.get("name", "") for a in paper.get("authors", [])),
                "year": paper.get("year"),
                "citations": paper.get("citationCount"),
                "abstract": abstract,
                "paperId": paper_id,
                "influential_citations": influential
            })

Paper error: {"message": "Too Many Requests. Please wait and try again or apply for a key for higher rate limits. https://www.semanticscholar.org/product/api#api-key-form", "code": "429"}
Paper error: {"message": "Too Many Requests. Please wait and try again or apply for a key for higher rate limits. https://www.semanticscholar.org/product/api#api-key-form", "code": "429"}
Paper error: {"message": "Too Many Requests. Please wait and try again or apply for a key for higher rate limits. https://www.semanticscholar.org/product/api#api-key-form", "code": "429"}
Paper error: {"message": "Too Many Requests. Please wait and try again or apply for a key for higher rate limits. https://www.semanticscholar.org/product/api#api-key-form", "code": "429"}


In [14]:
df = pd.DataFrame(all_rows)

In [15]:
df

,title,authors,year,citations,abstract,paperId,influential_citations
0,Judging LLM-as-a-judge with MT-Bench and Chatb...,"Lianmin Zheng, Wei-Lin Chiang, Ying Sheng, Siy...",2023.0,7772,Evaluating large language model (LLM) based ch...,a0a79dad89857a96f8f71b14238e5237cbfc4787,1050
1,DAPO: An Open-Source LLM Reinforcement Learnin...,"Qiying Yu, Zheng Zhang, Ruofei Zhu, Yufeng Yua...",2025.0,1509,Inference scaling empowers LLMs with unprecede...,dd4cfde3e135f799a9a71b4f57e13a29de89f7e3,306
2,Scaling LLM Test-Time Compute Optimally can be...,"C. Snell, Jaehoon Lee, Kelvin Xu, Aviral Kumar",2024.0,1574,Enabling LLMs to improve their outputs by usin...,8292083dd8f6ae898ea0ee54a6b97997d1a51c9d,117
3,A Survey on LLM-as-a-Judge,"Jiawei Gu, Xuhui Jiang, Zhichao Shi, Hexiang T...",2024.0,1178,Accurate and consistent evaluation is crucial ...,e24424283c02fbe7f641e5b3490d7bb059f8355a,103
4,A-MEM: Agentic Memory for LLM Agents,"Wujiang Xu, Zujie Liang, Kai Mei, Hang Gao, Ju...",2025.0,374,While large language model (LLM) agents can ef...,1f35a15fe9df43d24ec6ea551ec6c9766c17eccf,57
...,...,...,...,...,...,...,...
1765,Semi-symbolic inference for efficient streamin...,"Eric Hamilton Atkinson, Charles Yuan, Guillaum...",2022.0,24,A streaming probabilistic program receives a s...,beb136430b98eb42445a65baab28d9a610fbabf4,2
1766,"DeepSeek-V2: A Strong, Economical, and Efficie...","Zhihong Shao, Damai Dai, Daya Guo, Bo Liu (Ben...",2024.0,1091,"We present DeepSeek-V2, a strong Mixture-of-Ex...",53a803388e83ae89261624099d7be4287ace67cb,108
1767,Corrigendum to: IQ-TREE 2: New Models and Effi...,"B. Q. Minh, H. Schmidt, O. Chernomor, Dominik ...",2020.0,389,NaN,154e4fab30f9ff2dcb657814d63ea6866c143e74,41
1768,NN-LUT: Neural Approximation of Non-Linear Ope...,"Joonsang Yu, Junki Park, Seongmin Park, Minsoo...",2021.0,83,"Non-linear operations such as GELU, Layer norm...",844dd81f30d27ab9e42e11a5abb532095b36ac06,8


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1770 entries, 0 to 1769
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   title                  1770 non-null   str    
 1   authors                1770 non-null   str    
 2   year                   1767 non-null   float64
 3   citations              1770 non-null   int64  
 4   abstract               1589 non-null   str    
 5   paperId                1770 non-null   str    
 6   influential_citations  1770 non-null   int64  
dtypes: float64(1), int64(2), str(4)
memory usage: 96.9 KB


In [17]:
df.to_csv("papers.csv", index=False)

In [21]:
import ollama

def generate_embeddings(df):
    df['embedding'] = df['abstract'].apply(lambda x: ollama.embed(model="embeddinggemma", input=x)["embeddings"][0])
    return df

# function to remove entries in dataframe with missing abstracts
def remove_missing_abstracts(df):
    return df.dropna(subset=['abstract']).reset_index(drop=True)

In [26]:
df = remove_missing_abstracts(df)

df_embed = generate_embeddings(df)

,title,authors,year,citations,abstract,paperId,influential_citations,embedding
0,Judging LLM-as-a-judge with MT-Bench and Chatb...,"Lianmin Zheng, Wei-Lin Chiang, Ying Sheng, Siy...",2023.0,7772,Evaluating large language model (LLM) based ch...,a0a79dad89857a96f8f71b14238e5237cbfc4787,1050,"[-0.036970366, 0.026041301, -0.032789417, 0.00..."
1,DAPO: An Open-Source LLM Reinforcement Learnin...,"Qiying Yu, Zheng Zhang, Ruofei Zhu, Yufeng Yua...",2025.0,1509,Inference scaling empowers LLMs with unprecede...,dd4cfde3e135f799a9a71b4f57e13a29de89f7e3,306,"[-0.050388854, 0.02316877, 0.05043329, -0.0167..."
2,Scaling LLM Test-Time Compute Optimally can be...,"C. Snell, Jaehoon Lee, Kelvin Xu, Aviral Kumar",2024.0,1574,Enabling LLMs to improve their outputs by usin...,8292083dd8f6ae898ea0ee54a6b97997d1a51c9d,117,"[-0.034153845, -0.024409253, -0.0020903517, -0..."
3,A Survey on LLM-as-a-Judge,"Jiawei Gu, Xuhui Jiang, Zhichao Shi, Hexiang T...",2024.0,1178,Accurate and consistent evaluation is crucial ...,e24424283c02fbe7f641e5b3490d7bb059f8355a,103,"[-0.029957604, 0.015542779, 0.0019961877, 0.00..."
4,A-MEM: Agentic Memory for LLM Agents,"Wujiang Xu, Zujie Liang, Kai Mei, Hang Gao, Ju...",2025.0,374,While large language model (LLM) agents can ef...,1f35a15fe9df43d24ec6ea551ec6c9766c17eccf,57,"[-0.0762095, -0.016810821, -0.030484399, 0.000..."
...,...,...,...,...,...,...,...,...
1584,PUMA: A Programmable Ultra-efficient Memristor...,"Aayush Ankit, I. E. Hajj, S. R. Chalamalasetti...",2019.0,465,Memristor crossbars are circuits capable of pe...,d74bdbc0bce8ff90e815c74368cdac49b0eb4185,55,"[-0.051068373, 0.011437814, 0.008266899, -0.01..."
1585,Semi-symbolic inference for efficient streamin...,"Eric Hamilton Atkinson, Charles Yuan, Guillaum...",2022.0,24,A streaming probabilistic program receives a s...,beb136430b98eb42445a65baab28d9a610fbabf4,2,"[-0.079590105, -0.022412064, -0.013971629, 0.0..."
1586,"DeepSeek-V2: A Strong, Economical, and Efficie...","Zhihong Shao, Damai Dai, Daya Guo, Bo Liu (Ben...",2024.0,1091,"We present DeepSeek-V2, a strong Mixture-of-Ex...",53a803388e83ae89261624099d7be4287ace67cb,108,"[-0.031702068, 0.020593202, 0.111836396, 0.036..."
1587,NN-LUT: Neural Approximation of Non-Linear Ope...,"Joonsang Yu, Junki Park, Seongmin Park, Minsoo...",2021.0,83,"Non-linear operations such as GELU, Layer norm...",844dd81f30d27ab9e42e11a5abb532095b36ac06,8,"[-0.060237434, 0.01745827, 0.02377063, 0.01820..."


In [25]:
remove_missing_abstracts(df).info()

<class 'pandas.DataFrame'>
RangeIndex: 1589 entries, 0 to 1588
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   title                  1589 non-null   str    
 1   authors                1589 non-null   str    
 2   year                   1589 non-null   float64
 3   citations              1589 non-null   int64  
 4   abstract               1589 non-null   str    
 5   paperId                1589 non-null   str    
 6   influential_citations  1589 non-null   int64  
dtypes: float64(1), int64(2), str(4)
memory usage: 87.0 KB
